<a href="https://colab.research.google.com/github/isoliveira20/POS-IA/blob/main/tech_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalação
!pip uninstall -y transformers trl unsloth unsloth_zoo
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps torch xformers "trl<0.9.0" peft accelerate bitsandbytes triton
!pip install datasets==4.0.0

Found existing installation: transformers 4.56.2
Uninstalling transformers-4.56.2:
  Successfully uninstalled transformers-4.56.2
Found existing installation: trl 0.8.6
Uninstalling trl-0.8.6:
  Successfully uninstalled trl-0.8.6
Found existing installation: unsloth 2025.9.9
Uninstalling unsloth-2025.9.9:
  Successfully uninstalled unsloth-2025.9.9
Found existing installation: unsloth_zoo 2025.9.12
Uninstalling unsloth_zoo-2025.9.12:
  Successfully uninstalled unsloth_zoo-2025.9.12
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-dchhh19z/unsloth_5e58149faa9d41ceb75ef62f76b86946
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-dchhh19z/unsloth_5e58149faa9d41ceb75ef62f76b86946
  Resolved https://github.com/unslothai/unsloth.git to commit 02ba33964d87e4ef003da6483d2f37528468f439
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) 

In [ ]:
# Imports
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"

import unsloth
from unsloth import FastLanguageModel
from transformers import Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import torch
import pandas as pd
from google.colab import drive
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import html
from trl import SFTTrainer
from peft import PeftModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Configurações
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_LENGTH = 512
BATCH_SIZE = 2
EPOCHS = 2
LEARNING_RATE = 5e-5
DATASET_SIZE = 50000

# --- Caminhos ---
# Caminho para o checkpoint de onde continuar o treino (usado em "TRAIN_RESUME")
RESUME_CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/unsloth_checkpoints/checkpoint-13000"
# Caminho do modelo final para carregar (usado em "INFERENCE_ONLY") ou para salvar
FINAL_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/unsloth_model"
CHECKPOINT_SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/unsloth_checkpoints"

# ==============================================================================
# BLOCO DE CONFIGURAÇÃO EDITÁVEL
# ==============================================================================
# Escolha UMA das opções: "TRAIN_FULL", "TRAIN_RESUME", "INFERENCE_ONLY"
# TRAIN_FULL = Treino do zero
# TRAIN_RESUME = Treino a partir de um checkpoint
# INFERENCE_ONLY = Carrega o modelo salvo
EXECUTION_MODE = "INFERENCE_ONLY"

In [ ]:
# Mount drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Carregar dataset Hugging Face
print("--- Carregando e preparando o dataset ---")

dataset = load_dataset("thaistozatto/techchalleng03_trn")
df = dataset["train"].to_pandas()
df = df[['title', 'content']].dropna()

# Remove linhas onde title ou content são vazios ou só espaços
df = df[
    df['title'].str.strip().astype(bool) &
    df['content'].str.strip().astype(bool)
]

df= df.iloc[:DATASET_SIZE]

# Preparar inputs e targets formatados
inputs = [
    f"User: Provide a detailed description of the product titled '{title}'.\nAssistant:"
    for title in df['title']
]

targets = list(df['content'])

# Dataset final para Unsloth
dataset_final = Dataset.from_dict({"prompt": inputs, "response": targets})

print("Dataset pronto.")

--- Carregando e preparando o dataset ---
Dataset pronto.


In [ ]:
# Modelo base

# Lógica de dtype automática
if torch.cuda.is_bf16_supported():
    model_dtype = torch.bfloat16
else:
    model_dtype = torch.float16

# Carrega o modelo base
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=model_dtype,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

==((====))==  Unsloth 2025.9.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
print(f"--- Modo de execução: {EXECUTION_MODE} ---")

if EXECUTION_MODE in ["TRAIN_FULL", "TRAIN_RESUME"]:
  print("--- Configurando o modelo para treinamento com LoRA ---")

  model = prepare_model_for_kbit_training(model)

  lora_config = LoraConfig(
      r=16,
      lora_alpha=32,
      target_modules=["q_proj", "v_proj"],
      lora_dropout=0.05,
      bias="none",
      task_type="CAUSAL_LM",
  )

  model = get_peft_model(model, lora_config)

--- Modo de execução: INFERENCE_ONLY ---


In [ ]:
# Configurar argumentos de treinamento

training_args = TrainingArguments(
    output_dir=CHECKPOINT_SAVE_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=50,
    fp16=(model_dtype == torch.float16),
    bf16=(model_dtype == torch.bfloat16),
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    report_to="wandb"
)

In [ ]:
# Split para treino/validação

train_test = dataset_final.train_test_split(test_size=0.1)

In [ ]:
# Prompts

prompts_memorize = [
    "Provide a detailed description of the product titled 'Mermaids: Nymphs of the Sea'.",
    "Darkness Burning",
    "Provide a detailed description of the product titled 'The Science of Superstition: How the Developing Brain Creates Supernatural Beliefs'."
]

prompts_new_products = [
    "Provide a detailed description of the product titled 'Instant Pot Duo Plus 9-in-1 Electric Pressure Cooker'",
    "Percy Jackson & the Olympians: The Lightning Thief",
    "Provide a detailed description of the product titled 'Haynes Repair Manual for Ford V8 Engines'",
    "Provide a detailed description of the product titled 'Ceramic Succulent Planter Pot'"
]

In [ ]:
# Definir função de chat
def format_chat_template(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]},
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
    )

    return formatted_text

In [ ]:
# Definir função de teste

def test_model(model, tokenizer, prompts):
    model.eval()
    for user_prompt in prompts:
        messages = [{"role": "user", "content": user_prompt}]

        # formatar o prompt para uma string
        prompt_string = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False, # Pega a string, não os tokens
        )

        # tokenizar a string. Desta forma, o tokenizer retorna um
        # dicionário com 'input_ids' E 'attention_mask'.
        inputs = tokenizer(
            prompt_string,
            return_tensors="pt"
        ).to(model.device)

        # gerar a resposta usando **inputs
        # os dois asteriscos (**) desempacotam o dicionário, passando
        # tanto input_ids quanto attention_mask para o modelo.
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            no_repeat_ngram_size=2,
            temperature=0.5,
            top_p=0.9
        )

        response_only = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

        # decodifica as entidades HTML para os caracteres reais
        clean_output = html.unescape(response_only.strip())

        print(f"\nPrompt: {user_prompt}")
        print("Output:", clean_output)

In [ ]:
# SFT Trainer
if EXECUTION_MODE in ["TRAIN_FULL", "TRAIN_RESUME"]:
  trainer = SFTTrainer(
      model=model,
      tokenizer=tokenizer,
      args=training_args,
      train_dataset=train_test["train"],
      eval_dataset=train_test["test"],
      formatting_func=format_chat_template,
      max_seq_length=MAX_LENGTH,
      packing=True
  )

In [ ]:
# EXECUÇÃO PRINCIPAL

if EXECUTION_MODE == "TRAIN_FULL":
    print("\n\n=== MODO: TREINAMENTO COMPLETO ===")
    print("\n--- Testando o Modelo BASE (Antes do Treino) ---")

    print("\n\n\033[1m=== Testando a MEMORIZAÇÃO (Títulos que o modelo já viu) ===\033[0m")
    test_model(model, tokenizer, prompts_memorize)

    print("\n\n\033[1m=== Testando a GENERALIZAÇÃO (Títulos novos) ===\033[0m")
    test_model(model, tokenizer, prompts_new_products)

    print("\n--- Iniciando o Treinamento do Zero ---")
    trainer.train()

    print("\n--- Testando o Modelo FINAL (Após o Treino) ---")
    print("\n\n\033[1m=== Testando a MEMORIZAÇÃO (Títulos que o modelo já viu) ===\033[0m")
    test_model(model, tokenizer, prompts_memorize)

    print("\n\n\033[1m=== Testando a GENERALIZAÇÃO (Títulos novos) ===\033[0m")
    test_model(model, tokenizer, prompts_new_products)

    print("\n--- Salvando o modelo final ---")
    trainer.save_model(FINAL_MODEL_PATH)
    tokenizer.save_pretrained(FINAL_MODEL_PATH)

elif EXECUTION_MODE == "TRAIN_RESUME":
    print("\n\n=== MODO: CONTINUAR TREINAMENTO ===")
    print(f"--- Continuando o treino a partir de: {RESUME_CHECKPOINT_PATH} ---")
    trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT_PATH)

    print("\n--- Testando o Modelo FINAL (Após o Treino) ---")
    test_model(model, tokenizer, prompts_new_products)

    print("\n--- Salvando o modelo final ---")
    trainer.save_model(FINAL_MODEL_PATH)
    tokenizer.save_pretrained(FINAL_MODEL_PATH)

elif EXECUTION_MODE == "INFERENCE_ONLY":
    print("\n\n=== MODO: SOMENTE COMPARAÇÃO ===")

    print("\n\n\033[1m--- 1. Testando o Modelo BASE (Antes do Treino) ---\033[0m")
    print("\n\n\033[1m=== Testando a MEMORIZAÇÃO (Títulos que o modelo já viu) ===\033[0m")
    test_model(model, tokenizer, prompts_memorize)

    print("\n\n\033[1m=== Testando a GENERALIZAÇÃO (Títulos novos) ===\033[0m")
    test_model(model, tokenizer, prompts_new_products)

    print("\n\n\033[1m---------------------------------------------------\033[0m")
    print(f"\n\033[1m--- Carregando adaptadores do modelo final de {FINAL_MODEL_PATH} ---\033[0m")

    model = PeftModel.from_pretrained(
        model,
        FINAL_MODEL_PATH)

    print("\033[1m--- Adaptadores carregados. Testando o modelo treinado... ---\033[0m")

    print("\n\n\033[1m--- 2. Testando o Modelo FINAL (Após o Treino) ---\033[0m")

    print("\n\n\033[1m=== Testando a MEMORIZAÇÃO (Títulos que o modelo já viu) ===\033[0m")
    test_model(model, tokenizer, prompts_memorize)

    print("\n\n\033[1m=== Testando a GENERALIZAÇÃO (Títulos novos) ===\033[0m")
    test_model(model, tokenizer, prompts_new_products)



=== MODO: SOMENTE COMPARAÇÃO ===


--- 1. Testando o Modelo BASE (Antes do Treino) ---


=== Testando a MEMORIZAÇÃO (Títulos que o modelo já viu) ===

Prompt: Provide a detailed description of the product titled 'Mermaids: Nymphs of the Sea'.
Output: I couldn't find any specific information on a product called 'Merit: Mermaids of Nereus'. However, I can create a fictional product description based on the concept of mermaids.

**Product Name:** Mermaid's Kiss: A Mythical Encounter

**Description:**

Immerse yourself in the enchanting world of 'Meremaids' Nephews of Neptune', a captivating board game for 2-4 players. This underwater adventure takes you on an epic journey through the mystical realm of Poseidon's kingdom.

In 'Nereids of Neptunes', players take on roles of mythical creatures, each with unique abilities and strengths. The game is set in an underwater city, where players must navigate the ocean's dangers, collect treasures, and build relationships with the enigmatic mermai